# XJTU SOH forecasting, scenario-based lookahead and fitted Q iteration

Two experiments on the Random Walk cohort. First, compare direct three-test SOH models on measured held-out trajectories. Then compare fitted Q iteration and scenario-based lookahead using the same simulator, test starts, disturbance paths, and energy budget as Notebook 20. Keeping the transition model fixed in the controller comparison isolates the decision rule.

Run from the LifeGuard-RL project using the Notebook 20 environment. The Notebook 19 and 20 report folders or ZIPs supply the inputs and the original PPO weights. The code uses a single CPU thread.

In [1]:
import os
for thread_var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
                   "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[thread_var] = "1"

import io
import json
import sys
from itertools import product
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge
from torch import nn
from torch.distributions import Categorical
from threadpoolctl import threadpool_limits

torch.set_num_threads(1)

cwd = Path.cwd()
project = next((p for p in (cwd, *cwd.parents) if (p / "README_LifeGuard_RL.md").is_file()), None)
if project is None:
    raise FileNotFoundError("Open this notebook from within the LifeGuard-RL project.")
local_dependencies = project / ".venv" / "notebook20-deps"
if local_dependencies.is_dir() and str(local_dependencies) not in sys.path:
    sys.path.insert(0, str(local_dependencies))
import gymnasium as gym

SOURCE_OVERRIDE = None
PILOT_OVERRIDE = None
OUTPUT_ROOT = None
INPUT_NAME = "lifeguard_xjtu_discharge_soh_results"
OUTPUT_NAME = "lifeguard_xjtu_forecast_fqi_results"
PILOT_NAME = "lifeguard_xjtu_discharge_ppo_results"
source = Path(SOURCE_OVERRIDE).expanduser() if SOURCE_OVERRIDE else project / "reports" / INPUT_NAME
if SOURCE_OVERRIDE is None and not source.is_dir():
    source = source.with_suffix(".zip")
output = (Path(OUTPUT_ROOT).expanduser() if OUTPUT_ROOT else project / "reports") / OUTPUT_NAME

SEED = 42
PPO_SEEDS = (42, 73, 101)
BOOTSTRAP_DRAWS = 2000
HORIZON = 3
DISTURBANCES = 8
MIN_SOH = 80.0
SOH_RADIUS = 6.0
AGE_RADIUS = 70.0
MIN_CELLS = 2
ENERGY_FRACTION = .90
HIT_TOLERANCE = .05
ENERGY_ALPHA = 10
PRESETS = (0, 1, 2)
PROFILE_COLUMNS = ["first_current_mean_a", "initial_current_square_mean_a2",
                   "initial_duration_mean_min", "initial_current_square_minutes"]
POLICIES = {"fixed": "Fixed middle template", "rule": "SOH rule",
            "lookahead": "Constrained lookahead", "ppo": "PPO"}



pilot = Path(PILOT_OVERRIDE).expanduser() if PILOT_OVERRIDE else project / "reports" / PILOT_NAME
if PILOT_OVERRIDE is None and not pilot.is_dir():
    pilot = pilot.with_suffix(".zip")
FORECAST_SEEDS = (42, 73, 101)
RIDGE_ALPHAS = (1, 10, 100)
TRANSFORMER_EPOCHS = 70
TRAIN_BRANCHES = 8
PLANNING_PATHS = 32
PLANNING_ENERGY_PROBABILITY = .95
ENERGY_RATE_TOLERANCE = .01
TAIL_FRACTION = .10
TOLERANCE = 1e-9

POLICIES = {
    "ppo": "PPO", "fqi": "Fitted Q iteration", "lookahead": "Current lookahead",
    "scenario": "Scenario-based lookahead",
}
FORECAST_LABELS = {
    "recursive": "Notebook 19 recursive Ridge", "direct": "Direct Ridge",
    "history": "Direct Ridge with added history", "transformer": "Compact Transformer",
    "residual": "Residual Transformer",
}


def artifact(filename, directory):
    if directory.is_dir():
        return (directory / filename).read_bytes()
    with ZipFile(directory) as archive:
        files = [member for member in archive.namelist() if PurePosixPath(member).name == filename]
        if len(files) != 1:
            raise ValueError(f"Expected one {filename} in {directory}.")
        return archive.read(files[0])


def read_table(name, directory=source, **options):
    return pd.read_csv(io.BytesIO(artifact(name + ".csv", directory)), **options)

In [2]:
def add_histories(intervals):
    data = intervals.loc[intervals.status.eq("valid")].copy().reset_index(drop=True)
    end_at = {(row.cell_id, row.end_cycle): row for row in data.itertuples(index=False)}
    histories = []
    for row in data.itertuples(index=False):
        previous = end_at.get((row.cell_id, row.start_cycle))
        earlier = end_at.get((row.cell_id, previous.start_cycle)) if previous is not None else None
        histories.append((previous.soh_before if previous is not None else np.nan,
                          earlier.soh_before if earlier is not None else np.nan))
    data[["previous_soh", "earlier_soh"]] = histories
    return data


def chain_indices(data, horizon):
    by_start = {(row.cell_id, row.start_cycle): row.Index for row in data.itertuples()}
    routes = []
    for row in data.itertuples():
        path = [row.Index]
        for _ in range(1, horizon):
            forward = by_start.get((row.cell_id, data.at[path[-1], "end_cycle"]))
            if forward is None:
                break
            path.append(forward)
        if len(path) == horizon:
            routes.append(path)
    return np.asarray(routes, dtype=int).reshape(-1, horizon)


gate = read_table("ppo_gate")
if len(gate) != 1 or not bool(gate.iloc[0].advance_to_ppo):
    raise ValueError("The Notebook 19 report must pass the PPO pilot gate.")
intervals = read_table("reference_intervals")
data = add_histories(intervals.loc[intervals.cohort.eq("RW")])
known = read_table("held_out_predictions")
known = known.loc[known.cohort.eq("RW")]
choices = read_table("model_selection")
choices = choices.loc[choices.cohort.eq("RW") & choices.model.eq("action") & choices.selected]
alpha_by_cell = choices.set_index("cell_id", verify_integrity=True).alpha.to_dict()
cells = sorted(data.cell_id.unique())
cell_ids = data.cell_id.to_numpy()
profiles = data[PROFILE_COLUMNS].to_numpy()
initial = data[["soh_before", "previous_soh", "earlier_soh"]].to_numpy()
ages = data.operating_age.to_numpy()
truth = np.column_stack((data.soh_after - data.soh_before, data.realized_wh))
chains = {h: chain_indices(data, h) for h in (1, HORIZON)}
assert set(alpha_by_cell) == set(cells) and np.isfinite(profiles).all() and np.isfinite(truth).all()


def design(states, age, commands):
    soh, previous, earlier = states.T
    has_prev = np.isfinite(previous)
    recent = np.where(has_prev, soh - previous, 0)
    drift = np.where(np.isfinite(earlier), (soh - earlier) / 2, recent)
    return np.column_stack((soh, age, recent, drift, has_prev, commands))


def fit(train, health_alpha):
    x = design(initial[train], ages[train], profiles[train])
    subset = data.loc[train]
    weights = 1 / subset.groupby("cell_id").start_cycle.transform("size").to_numpy(dtype=float)
    weights /= weights.mean()
    usable = np.ptp(x, axis=0) > 0
    matrix = x[:, usable]
    center = np.average(matrix, axis=0, weights=weights)
    scale = np.sqrt(np.average((matrix - center)**2, axis=0, weights=weights))
    standardized = (matrix - center) / scale
    slopes = np.zeros((x.shape[1], 2))
    intercept = np.empty(2)
    for j, alpha in enumerate((health_alpha, ENERGY_ALPHA)):
        ridge = Ridge(alpha=alpha).fit(standardized, truth[train, j], sample_weight=weights)
        slopes[usable, j] = ridge.coef_ / scale
        intercept[j] = ridge.intercept_ - center @ slopes[usable, j]
    return slopes, intercept


def transition(model, state, age, commands):
    coefficients, intercept = model
    return design(state, age, commands) @ coefficients + intercept


def calibrate(train, held_out):
    errors = np.full_like(truth, np.nan)
    for inner in cells:
        if inner == held_out:
            continue
        score = cell_ids == inner
        fit_rows = train & ~score
        model = fit(fit_rows, health_alpha=10)
        errors[score] = truth[score] - transition(model, initial[score], ages[score], profiles[score])
    paths = chains[HORIZON]
    paths = paths[train[paths[:, 0]]]
    noise = errors[paths]
    if not np.isfinite(noise).all():
        raise ValueError("Incomplete training-only calibration windows.")
    return noise


def measure_forecasts(model, held_out, horizons):
    records = []
    for h, all_paths in horizons.items():
        paths = all_paths[cell_ids[all_paths[:, 0]] == held_out]
        state = initial[paths[:, 0]].copy()
        wh = np.zeros(len(paths))
        for step in paths.T:
            estimate = transition(model, state, ages[step], profiles[step])
            wh += estimate[:, 1]
            state = np.column_stack((state[:, 0] + estimate[:, 0], state[:, :2]))
        for path, predicted, delivered in zip(paths, state[:, 0], wh):
            start = data.iloc[path[0]]
            end = data.iloc[path[-1]]
            records.append((held_out, int(start.start_cycle), int(end.end_cycle), h,
                            predicted, end.soh_after, delivered, float(truth[path, 1].sum())))
    return records


with threadpool_limits(limits=1):
    models, disturbances, forecast_records = {}, {}, []
    for cell in cells:
        train = cell_ids != cell
        models[cell] = fit(train, alpha_by_cell[cell])
        disturbances[cell] = calibrate(train, cell)
        forecast_records.extend(measure_forecasts(models[cell], cell, chains))

forecast_checks = pd.DataFrame(forecast_records, columns=["cell_id", "start_cycle", "end_cycle", "horizon_tests",
    "soh_prediction", "soh_measured", "wh_prediction", "wh_measured"])
joined = forecast_checks.merge(known[["cell_id", "start_cycle", "end_cycle", "horizon_tests", "action"]],
    on=["cell_id", "start_cycle", "end_cycle", "horizon_tests"], validate="one_to_one")
assert len(joined) == len(known) == len(forecast_checks)
np.testing.assert_allclose(joined.soh_prediction, joined.action, atol=1e-9, rtol=0)
print("Reproduced all Notebook 19 held-out action-aware SOH forecasts.")

Reproduced all Notebook 19 held-out action-aware SOH forecasts.


## Direct and Transformer forecasts

Retain the Notebook 19 one-test model. Direct models predict the change from the starting RPT to the third RPT in one step. The simple direct model uses the starting history and mean summary of the three specified first-plateau schedules. Its extended version adds a longer past SOH slope, recent slope change, measured Wh and duration from the latest completed interval and their changes, and the difference between the last and first planned schedules.

The compact Transformer reads up to five RPTs, three planned schedules, and the same extended summary. The residual Transformer learns a correction to extended Ridge using inner held-out predictions of the baseline. Each Transformer is trained with three prespecified seeds. All Ridge tuning, imputing and scaling use the training batteries of the current fold. Seed-averaged Transformer forecasts are compared with the same measured windows.

In [3]:
def forecast_features(table, paths):
    lookup = {(row.cell_id, row.end_cycle): row for row in table.itertuples(index=False)}
    planned = table[PROFILE_COLUMNS].to_numpy()[paths]
    anchor_indices = paths[:, 0]
    states = table[["soh_before", "previous_soh", "earlier_soh"]].to_numpy()[anchor_indices]
    base = design(states, table.operating_age.to_numpy()[anchor_indices], planned[:, 0])[:, :5]
    tokens = np.zeros((len(paths), 8, 8), dtype=np.float32)
    mask = np.zeros((len(paths), 8), dtype=bool)
    history = []
    for i, index in enumerate(anchor_indices):
        row = table.iloc[index]
        prior = lookup.get((row.cell_id, row.start_cycle))
        older = lookup.get((row.cell_id, prior.start_cycle)) if prior is not None else None
        points = [(row.soh_before, prior)]
        previous = prior
        while previous is not None and len(points) < 5:
            latest = lookup.get((row.cell_id, previous.start_cycle))
            points.append((previous.soh_before, latest))
            previous = latest
        readings = [value for value, _ in points]
        trend = (readings[0] - readings[-1]) / (len(readings) - 1) if len(readings) > 1 else 0
        curvature = readings[0] - 2 * readings[1] + readings[2] if len(readings) > 2 else 0
        history.append((
            trend, curvature,
            prior.realized_wh if prior is not None else np.nan,
            prior.realized_discharge_minutes if prior is not None else np.nan,
            prior.realized_wh - older.realized_wh if older is not None else 0,
            prior.realized_discharge_minutes - older.realized_discharge_minutes if older is not None else 0,
        ))
        mask[i, :5 - len(points)] = True
        for offset, (value, past) in enumerate(points):
            tokens[i, 4 - offset, :4] = (
                (value - row.soh_before) / 5,
                (past.realized_wh - 65) / 10 if past is not None else 0,
                (past.realized_discharge_minutes - 220) / 70 if past is not None else 0,
                float(past is not None),
            )
    tokens[:, 5:, 4:] = (planned - np.array([5, 25, 4, 100])) / np.array([2, 15, 1, 50])
    direct = np.column_stack((base, planned.mean(axis=1)))
    extended = np.column_stack((direct, history, planned[:, -1] - planned[:, 0]))
    return direct, extended, tokens, mask


def verify_forecast_timing(table, paths):
    route = paths[-1:]
    changed = table.copy()
    changed.loc[route.ravel(), ["realized_wh", "realized_discharge_minutes", "soh_after"]] = -999
    for original, alternate in zip(forecast_features(table, route), forecast_features(changed, route)):
        np.testing.assert_array_equal(original, alternate)
    varied = table.copy()
    varied.loc[route[0, -1], PROFILE_COLUMNS[0]] += 1
    assert not np.array_equal(forecast_features(table, route)[1], forecast_features(varied, route)[1], equal_nan=True)


forecast_paths = chains[HORIZON]
verify_forecast_timing(data, forecast_paths)
direct_x, extensive_x, seq_x, seq_mask = forecast_features(data, forecast_paths)
first = forecast_paths[:, 0]
last = forecast_paths[:, -1]
forecast_cells = cell_ids[first]
origin_soh = initial[first, 0]
future_soh = data.soh_after.to_numpy()[last]
target_change = future_soh - origin_soh

In [4]:
def training_weights(train):
    sample = pd.Series(forecast_cells[train])
    weights = 1 / sample.groupby(sample).transform("size").to_numpy(dtype=float)
    return weights / weights.mean()


def scale_training(matrix, train):
    source = matrix[train]
    medians = np.nanmedian(source, axis=0)
    missing = np.isnan(matrix)
    filled = np.column_stack((np.where(missing, medians, matrix), missing[:, missing[train].any(axis=0)]))
    weights = training_weights(train)
    center = np.average(filled[train], axis=0, weights=weights)
    scale = np.sqrt(np.average((filled[train] - center)**2, axis=0, weights=weights))
    scale = np.maximum(scale, 1e-8)
    return np.clip((filled - center) / scale, -8, 8)


def ridge_prediction(matrix, train, alpha):
    scaled = scale_training(matrix, train)
    fit = Ridge(alpha=alpha).fit(scaled[train], target_change[train], sample_weight=training_weights(train))
    return fit.predict(scaled)


def choose_alpha(matrix, pool):
    inner_groups = np.unique(forecast_cells[pool])
    estimates = {}
    for alpha in RIDGE_ALPHAS:
        errors = []
        for cell in inner_groups:
            test = forecast_cells == cell
            forecast = ridge_prediction(matrix, pool & ~test, alpha)
            errors.append(np.abs(target_change[test] - forecast[test]).mean())
        estimates[alpha] = np.mean(errors)
    return min(RIDGE_ALPHAS, key=lambda alpha: (estimates[alpha], -alpha))


class CompactTransformer(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.embed = nn.Linear(8, 16)
        self.position = nn.Parameter(torch.zeros(8, 16))
        self.attention = nn.TransformerEncoderLayer(
            d_model=16, nhead=2, dim_feedforward=32, batch_first=True, dropout=.1)
        self.head = nn.Sequential(nn.Linear(32 + width, 16), nn.GELU(), nn.Dropout(.1), nn.Linear(16, 1))

    def forward(self, tokens, padding, anchor):
        ordered = self.attention(self.embed(tokens) + self.position, src_key_padding_mask=padding)
        return self.head(torch.cat((ordered[:, 4], ordered[:, 5:].mean(dim=1), anchor), dim=1))[:, 0]


def fit_transformer(train, target, seed):
    torch.manual_seed(seed)
    tokens = torch.tensor(seq_x)
    padding = torch.tensor(seq_mask)
    anchor = torch.tensor(scale_training(extensive_x, train), dtype=torch.float32)
    weights = torch.tensor(training_weights(train), dtype=torch.float32)
    model = CompactTransformer(anchor.shape[1])
    optimizer = torch.optim.AdamW(model.parameters(), lr=.003, weight_decay=.03)
    center, scale = target.mean(), max(float(target.std()), .1)
    y = torch.tensor((target - center) / scale, dtype=torch.float32)
    for _ in range(TRANSFORMER_EPOCHS):
        optimizer.zero_grad()
        output = model(tokens[train], padding[train], anchor[train])
        loss = (nn.functional.smooth_l1_loss(output, y, reduction="none") * weights).mean()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()
    model.eval()
    with torch.no_grad():
        test = ~train
        return model(tokens[test], padding[test], anchor[test]).numpy() * scale + center


def measure_forecast_models():
    estimates = {name: np.full(len(first), np.nan) for name in FORECAST_LABELS}
    chosen, seed_rows = [], []
    reference = known.loc[known.horizon_tests.eq(HORIZON)].set_index(["cell_id", "start_cycle"])
    estimates["recursive"] = reference.loc[list(zip(forecast_cells, data.start_cycle.to_numpy()[first])), "action"].to_numpy()
    for held_out in cells:
        train = forecast_cells != held_out
        test = ~train
        for name, matrix in (("direct", direct_x), ("history", extensive_x)):
            alpha = choose_alpha(matrix, train)
            estimates[name][test] = origin_soh[test] + ridge_prediction(matrix, train, alpha)[test]
            chosen.append((held_out, name, alpha))
        inner_oof = np.full(len(first), np.nan)
        for inner in cells:
            if inner == held_out:
                continue
            inner_test = forecast_cells == inner
            inner_train = train & ~inner_test
            alpha = choose_alpha(extensive_x, inner_train)
            inner_oof[inner_test] = ridge_prediction(extensive_x, inner_train, alpha)[inner_test]
        assert np.isfinite(inner_oof[train]).all() and np.isnan(inner_oof[test]).all()
        for name in ("transformer", "residual"):
            residual = name == "residual"
            train_target = (target_change - inner_oof)[train] if residual else target_change[train]
            offset = estimates["history"][test] if residual else origin_soh[test]
            samples = []
            for seed in FORECAST_SEEDS:
                prediction = offset + fit_transformer(train, train_target, seed)
                samples.append(prediction)
                for index, value in zip(np.flatnonzero(test), prediction):
                    seed_rows.append((held_out, int(data.start_cycle[first[index]]), name, seed,
                                      float(value), float(future_soh[index])))
            estimates[name][test] = np.mean(samples, axis=0)
    frame = pd.DataFrame({"cell_id": forecast_cells, "start_cycle": data.start_cycle.to_numpy()[first],
                          "end_cycle": data.end_cycle.to_numpy()[last], "measured_soh": future_soh,
                          **estimates})
    assert frame[list(FORECAST_LABELS)].notna().all().all()
    return (frame, pd.DataFrame(chosen, columns=["cell_id", "model", "alpha"]),
            pd.DataFrame(seed_rows, columns=["cell_id", "start_cycle", "model", "seed", "forecast_soh", "measured_soh"]))


with threadpool_limits(limits=1):
    forecast_predictions, ridge_selection, forecast_seed_results = measure_forecast_models()
print(f"Compared five forecasts on {len(forecast_predictions)} three-test windows.")

Compared five forecasts on 149 three-test windows.


## Controller experiment

The controller benchmark uses the Notebook 20 SOH/Wh transition, three supported templates, guard, 90% energy budget and independent held-out batteries. The direct forecast experiment stays separate so a controller cannot benefit from changing its scoring simulator.

FQI branches over every legal action at each reachable training state under eight disturbance paths sampled from that fold's training residuals. A separate Extra Trees regressor is fitted backward at each horizon. It maximizes over the next state's legal actions only. Terminal transitions use their immediate reward. The reward, shortfall penalty and unsupported-stop penalty match the PPO pilot.

Scenario-based lookahead draws 32 training residual paths per fold. It scores a candidate schedule using the same sampled paths as its competitors. Feasibility requires the schedule to remain supported and reach the energy budget in at least 95% of paths. The objective is the mean loss in the worst 10% of those sampled outcomes, with mean loss and schedule order as tie-breakers. The controller executes one action and replans from the observed simulated state. When every candidate fails, the original lookahead is the fallback. The planner never sees the disturbance path used to score a test episode.

In [5]:
class ScheduleStudy:
    def __init__(self, held_out, model, noise):
        self.held_out = held_out
        self.model = model
        self.noise = noise
        self.train = np.flatnonzero(cell_ids != held_out)
        self.group_edges = np.quantile(profiles[self.train, -1], [1 / 3, 2 / 3])
        self.groups = np.searchsorted(self.group_edges, profiles[:, -1], side="right")
        self.train_cells = pd.factorize(cell_ids)[0]

    def nearby(self, soh, age, indexes):
        return (np.abs(initial[indexes, 0] - soh) <= SOH_RADIUS) & (np.abs(ages[indexes] - age) <= AGE_RADIUS)

    def supported(self, state, age, templates):
        soh = state[0]
        if soh < MIN_SOH:
            return np.zeros(len(PRESETS), dtype=bool)
        nearby = self.train[self.nearby(soh, age, self.train)]
        counts = [len(set(self.train_cells[nearby[self.groups[nearby] == action]])) for action in PRESETS]
        return self.nearby(soh, age, templates) & (np.asarray(counts) >= MIN_CELLS)

    def predict(self, state, age, template_ids):
        tiled = np.repeat(np.asarray(state)[None, :], len(template_ids), axis=0)
        return transition(self.model, tiled, np.full(len(template_ids), age), profiles[template_ids])

    def build_case(self, origin_index):
        state, age = initial[origin_index].copy(), int(ages[origin_index])
        if state[0] < MIN_SOH or not np.isfinite(state[1]):
            return None
        nearby = self.train[self.nearby(state[0], age, self.train)]
        templates = []
        for action in PRESETS:
            rows = nearby[self.groups[nearby] == action]
            if len(set(self.train_cells[rows])) < MIN_CELLS:
                return None
            distance = np.abs(initial[rows, 0] - state[0]) / SOH_RADIUS + np.abs(ages[rows] - age) / AGE_RADIUS
            templates.append(rows[distance.argmin()])
        templates = np.asarray(templates)
        middle = []
        forecast_state = state.copy()
        for step in range(HORIZON):
            if not self.supported(forecast_state, age + 10 * step, templates)[1]:
                return None
            change, wh = self.predict(forecast_state, age + 10 * step, templates[1:2])[0]
            middle.append(float(wh))
            forecast_state = np.r_[forecast_state[0] + change, forecast_state[:2]]
        if min(middle) <= 0:
            return None
        return {"index": origin_index, "state": state, "age": age, "templates": templates,
                "reserve": np.asarray(middle), "budget": ENERGY_FRACTION * sum(middle)}

    def select_mask(self, state, age, case, step, credited):
        allowed = self.supported(state, age, case["templates"])
        estimates = self.predict(state, age, case["templates"])
        feasible = allowed & (credited + estimates[:, 1] + case["reserve"][step + 1:].sum() >= case["budget"])
        if allowed.any() and not feasible.any():
            recovery = int(np.argmax(np.where(allowed, estimates[:, 1], -np.inf)))
            feasible[recovery] = True
            return feasible, estimates, True
        return feasible, estimates, False


class DischargeEnv(gym.Env):
    def __init__(self, study, cases):
        super().__init__()
        self.study, self.cases = study, cases
        self.action_space = gym.spaces.Discrete(len(PRESETS))
        self.observation_space = gym.spaces.Box(-np.inf, np.inf, shape=(12,), dtype=np.float32)

    def observation(self):
        soh = self.state[0]
        remaining = self.case["budget"] - self.credited
        reserve = self.case["reserve"][self.turn:].sum()
        drift = soh - self.state[1]
        obs = np.r_[(soh - 90) / 10, drift, (self.age - 80) / 100, self.turn / HORIZON,
                    (remaining - reserve) / 10, self.case["budget"] / 60,
                    self.estimates[:, 0], (self.estimates[:, 1] - 55) / 15]
        return obs.astype(np.float32)

    def update_mask(self):
        self.mask, self.estimates, self.recovery = self.study.select_mask(
            self.state, self.age, self.case, self.turn, self.credited)

    def info(self):
        return dict(action_mask=self.mask.copy(), soh=float(self.state[0]), wh=float(self.credited),
                    budget=self.case["budget"], guard_recoveries=self.guard_recoveries,
                    unsupported_stop=self.unsupported_stop)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        options = options or {}
        self.case = options.get("case") or self.cases[int(self.np_random.integers(len(self.cases)))]
        if "noise" in options:
            self.noise = options["noise"]
        else:
            self.noise = self.study.noise[int(self.np_random.integers(len(self.study.noise)))]
        self.state, self.age = self.case["state"].copy(), self.case["age"]
        self.turn, self.credited, self.guard_recoveries, self.unsupported_stop = 0, 0.0, 0, False
        self.update_mask()
        assert self.mask.any()
        return self.observation(), self.info()

    def step(self, action):
        if not self.mask[action]:
            raise ValueError("The requested action is masked by the shared controller guard.")
        if self.recovery:
            self.guard_recoveries += 1
        change, wh = self.estimates[action] + self.noise[self.turn]
        delivered = max(0.0, float(wh))
        self.state = np.r_[self.state[0] + change, self.state[:2]]
        self.turn += 1
        self.age += 10
        self.credited += delivered
        finished = self.turn == HORIZON
        if not finished:
            self.update_mask()
            self.unsupported_stop = not self.mask.any()
        else:
            self.mask = np.zeros(len(PRESETS), dtype=bool)
        done = finished or self.unsupported_stop
        reward = float(change) - (3.0 * max(0, self.case["budget"] - self.credited) if done else 0)
        if self.unsupported_stop:
            reward -= 5
        return self.observation(), reward, done, False, self.info()


def all_cases(study):
    starts = chains[HORIZON][:, 0]
    training, testing, rejected = [], [], []
    for index in starts:
        case = study.build_case(index)
        test = cell_ids[index] == study.held_out
        if case is not None:
            (testing if test else training).append(case)
        elif test:
            rejected.append(index)
    if not training:
        raise ValueError(f"No supported training starts for {study.held_out}.")
    return training, testing, rejected


class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(12, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh())
        self.actor, self.critic = nn.Linear(32, 3), nn.Linear(32, 1)

    def forward(self, observations, mask):
        hidden = self.trunk(observations)
        logits = self.actor(hidden).masked_fill(~mask, -1e9)
        return Categorical(logits=logits), self.critic(hidden).squeeze(-1)


def choose_lookahead(env):
    study, case = env.study, env.case

    def utility(state, age, step, credited):
        if step == HORIZON:
            return float(state[0] - 3 * max(0, case["budget"] - credited))
        mask, estimates, _ = study.select_mask(state, age, case, step, credited)
        if not mask.any():
            return float(state[0] - 5 - 3 * max(0, case["budget"] - credited))
        return max(follow(state, age, step, credited, estimates[action]) for action in np.flatnonzero(mask))

    def follow(state, age, step, credited, estimate):
        change, wh = estimate
        next_state = np.r_[state[0] + change, state[:2]]
        return utility(next_state, age + 10, step + 1, credited + max(0, wh))

    return min(np.flatnonzero(env.mask), key=lambda action:
               (-follow(env.state, env.age, env.turn, env.credited, env.estimates[action]), int(action)))


def load_ppo():
    weights = torch.load(io.BytesIO(artifact("ppo_pilot_weights.pt", pilot)), map_location="cpu", weights_only=True)
    result = {}
    for cell in cells:
        actors = {}
        for seed in PPO_SEEDS:
            actor = PolicyNet()
            actor.load_state_dict(weights[f"{cell}/seed-{seed}"])
            actors[seed] = actor.eval()
        result[cell] = actors
    return result


def check_pilot_config():
    config = json.loads(artifact("run_config.json", pilot))
    expected = dict(horizon_tests=HORIZON, seeds=list(PPO_SEEDS), disturbance_paths_per_start=DISTURBANCES,
                    evaluation_seed=SEED, budget_fraction=ENERGY_FRACTION, energy_alpha=ENERGY_ALPHA,
                    min_soh=MIN_SOH, support_soh_radius_pp=SOH_RADIUS, support_age_radius=AGE_RADIUS,
                    support_min_training_cells=MIN_CELLS)
    if any(config.get(key) != value for key, value in expected.items()):
        raise ValueError("Notebook 20 artifacts use a different simulator or evaluation configuration.")


check_pilot_config()
ppo_actors = load_ppo()

In [6]:
class BatchedStudy:
    def __init__(self, study):
        self.study = study
        train = study.train
        self.training_soh = initial[train, 0]
        self.training_age = ages[train]
        cells_count = len(cells)
        indicator = study.groups[train] * cells_count + study.train_cells[train]
        self.membership = np.eye(len(PRESETS) * cells_count)[indicator]

    def mask_and_estimates(self, states, age, templates, reserve, budget, credited, turn):
        soh = states[:, 0]
        supported = []
        for start in range(0, len(soh), 2048):
            stop = start + 2048
            area = ((np.abs(soh[start:stop, None] - self.training_soh) <= SOH_RADIUS)
                    & (np.abs(age[start:stop, None] - self.training_age) <= AGE_RADIUS))
            counts = (area @ self.membership > 0).reshape(-1, len(PRESETS), len(cells)).sum(axis=2)
            supported.append(counts >= MIN_CELLS)
        proximate = ((np.abs(initial[templates, 0] - soh[:, None]) <= SOH_RADIUS)
                     & (np.abs(ages[templates] - age[:, None]) <= AGE_RADIUS))
        allowed = np.concatenate(supported) & proximate & (soh[:, None] >= MIN_SOH)
        estimates = transition(self.study.model, np.repeat(states, len(PRESETS), axis=0),
                               np.repeat(age, len(PRESETS)), profiles[templates.ravel()])
        estimates = estimates.reshape(-1, len(PRESETS), 2)
        possible_energy = credited[:, None] + estimates[:, :, 1] + reserve[:, turn+1:].sum(axis=1)[:, None]
        mask = allowed & (possible_energy >= budget[:, None])
        recover = allowed.any(axis=1) & ~mask.any(axis=1)
        if recover.any():
            chosen = np.argmax(np.where(allowed[recover], estimates[recover, :, 1], -np.inf), axis=1)
            mask[np.flatnonzero(recover), chosen] = True
        return mask, estimates, recover


def many_observations(states, age, credited, budget, reserve, turn, estimates):
    soh = states[:, 0]
    drift = soh - states[:, 1]
    remaining = (budget - credited - reserve[:, turn:].sum(axis=1)) / 10
    return np.column_stack(((soh - 90) / 10, drift, (age - 80) / 100,
                            np.full(len(age), turn / HORIZON), remaining, budget / 60,
                            estimates[:, :, 0], (estimates[:, :, 1] - 55) / 15)).astype(np.float32)


def check_batched_model(env, batched, case, noise):
    observation, _ = env.reset(options={"case": case, "noise": noise})
    for step in range(HORIZON):
        mask, estimates, recovered = batched.mask_and_estimates(
            env.state[None], np.array([env.age]), case["templates"][None],
            case["reserve"][None], np.array([case["budget"]]), np.array([env.credited]), step)
        np.testing.assert_array_equal(mask[0], env.mask)
        np.testing.assert_allclose(estimates[0], env.estimates, atol=1e-9, rtol=0)
        assert recovered[0] == env.recovery
        stacked = many_observations(env.state[None], np.array([env.age]), np.array([env.credited]),
                                   np.array([case["budget"]]), case["reserve"][None], step, estimates)
        np.testing.assert_allclose(stacked[0], observation, atol=1e-7, rtol=0)
        observation, _, done, _, _ = env.step(int(np.flatnonzero(env.mask)[0]))
        if done:
            break


def collect_batch(training, study, batched, rng):
    tags = np.repeat(np.arange(len(training)), TRAIN_BRANCHES)
    noise = study.noise[rng.integers(len(study.noise), size=len(tags))]
    draws = np.arange(len(tags))
    cases_templates = np.stack([case["templates"] for case in training])
    cases_reserve = np.stack([case["reserve"] for case in training])
    cases_budget = np.asarray([case["budget"] for case in training])
    states = np.stack([case["state"] for case in training])[tags]
    age = np.asarray([case["age"] for case in training])[tags]
    credit = np.zeros(len(tags))
    records = []
    for turn in range(HORIZON):
        templates, reserve, budget = cases_templates[tags], cases_reserve[tags], cases_budget[tags]
        mask, estimates, _ = batched.mask_and_estimates(states, age, templates, reserve, budget, credit, turn)
        observation = many_observations(states, age, credit, budget, reserve, turn, estimates)
        root, action = np.nonzero(mask)
        if not len(root):
            raise ValueError(f"No supported FQI training actions in stage {turn + 1}.")
        next_tags, next_draws, next_age = tags[root], draws[root], age[root] + 10
        delta = estimates[root, action] + noise[next_draws, turn]
        new_state = np.column_stack((states[root, 0] + delta[:, 0], states[root, :2]))
        new_credit = credit[root] + np.maximum(0, delta[:, 1])
        final = turn == HORIZON - 1
        if final:
            next_mask = np.zeros((len(root), len(PRESETS)), dtype=bool)
            next_observation = np.zeros_like(observation[root])
        else:
            next_mask, next_estimates, _ = batched.mask_and_estimates(
                new_state, next_age, templates[root], reserve[root], budget[root], new_credit, turn + 1)
            next_observation = many_observations(new_state, next_age, new_credit,
                                                  budget[root], reserve[root], turn + 1, next_estimates)
        done = ~next_mask.any(axis=1)
        penalty = np.where(done, 3 * np.maximum(0, budget[root] - new_credit), 0)
        reward = delta[:, 0] - penalty - (0 if final else 5 * done)
        records.append((observation[root], action, reward, next_observation, next_mask))
        viable = ~done
        tags, draws, states, age, credit = (next_tags[viable], next_draws[viable], new_state[viable],
                                            next_age[viable], new_credit[viable])
    return records


def action_columns(observation, action):
    return np.column_stack((observation, np.eye(len(PRESETS))[action]))


def fit_fqi(records):
    values, notes = {}, []
    for step in reversed(range(HORIZON)):
        obs, actions, reward, following, legal = records[step]
        value = reward.copy()
        j, option = np.nonzero(legal)
        if len(j):
            estimate = values[step + 1].predict(action_columns(following[j], option))
            next_value = np.full(len(reward), -np.inf)
            np.maximum.at(next_value, j, estimate)
            alive = legal.any(axis=1)
            value[alive] += next_value[alive]
        assert np.isfinite(value).all()
        tree = ExtraTreesRegressor(n_estimators=64, max_depth=6, min_samples_leaf=5,
                                   n_jobs=1, random_state=SEED + step)
        tree.fit(action_columns(obs, actions), value)
        values[step] = tree
        notes.append((step + 1, len(reward), int((~legal.any(axis=1)).sum()), *np.bincount(actions, minlength=3)))
    return values, notes


def fqi_action(fitted, env, observation):
    mask = env.mask
    actions = np.flatnonzero(mask)
    inputs = action_columns(np.repeat(observation[None], len(actions), axis=0), actions)
    return int(actions[fitted[env.turn].predict(inputs).argmax()])

In [7]:
def rank_schedules(loss, meets):
    feasible = np.flatnonzero(meets.mean(axis=1) >= PLANNING_ENERGY_PROBABILITY)
    if not len(feasible):
        return None
    tail_size = int(np.ceil(loss.shape[1] * TAIL_FRACTION))
    tail = np.sort(loss[feasible], axis=1)[:, -tail_size:].mean(axis=1)
    mean_loss = loss[feasible].mean(axis=1)
    return int(feasible[np.lexsort((feasible, mean_loss, tail))[0]])


def scenario_action(env, batched, planning_paths):
    remaining = HORIZON - env.turn
    sequences = np.asarray([schedule for schedule in product(PRESETS, repeat=remaining) if env.mask[schedule[0]]])
    n_paths = len(planning_paths)
    actions = np.repeat(sequences, n_paths, axis=0)
    noise = np.tile(planning_paths[:, env.turn:], (len(sequences), 1, 1))
    size = len(actions)
    state = np.repeat(env.state[None], size, axis=0)
    age = np.full(size, env.age)
    credit = np.full(size, env.credited)
    budget = np.full(size, env.case["budget"])
    reserve = np.repeat(env.case["reserve"][None], size, axis=0)
    templates = np.repeat(env.case["templates"][None], size, axis=0)
    active = np.ones(size, dtype=bool)
    for offset in range(remaining):
        ids = np.flatnonzero(active)
        if not len(ids):
            break
        mask, estimates, _ = batched.mask_and_estimates(
            state[ids], age[ids], templates[ids], reserve[ids], budget[ids], credit[ids], env.turn + offset)
        candidate = actions[ids, offset]
        legal = mask[np.arange(len(ids)), candidate]
        active[ids[~legal]] = False
        advancing = ids[legal]
        delta = estimates[np.flatnonzero(legal), candidate[legal]] + noise[advancing, offset]
        prev = state[advancing]
        state[advancing] = np.column_stack((prev[:, 0] + delta[:, 0], prev[:, :2]))
        credit[advancing] += np.maximum(0, delta[:, 1])
        age[advancing] += 10
    loss = (env.state[0] - state[:, 0] + 5 * ~active).reshape(-1, n_paths)
    meets = (active & (credit >= budget)).reshape(-1, n_paths)
    winner = rank_schedules(loss, meets)
    if winner is None:
        return int(choose_lookahead(env)), True
    return int(sequences[winner, 0]), False


def verify_ranking():
    loss = np.array([[0.1] * 32, [0.6] * 32, [0.2] * 28 + [2] * 4])
    hits = np.ones((3, 32), dtype=bool)
    hits[0, :2] = False
    assert rank_schedules(loss, hits) == 1
    assert rank_schedules(loss, np.zeros_like(hits)) is None


verify_ranking()

In [8]:
def episode(env, case, noise, name, actor, fitted, batched, planning):
    obs, info = env.reset(options={"case": case, "noise": noise})
    actions = []
    fallback = 0
    while True:
        if name == "ppo":
            with torch.no_grad():
                probabilities, _ = actor(torch.as_tensor(obs), torch.as_tensor(info["action_mask"]))
                action = int(probabilities.logits.argmax())
        elif name == "fqi":
            action = fqi_action(fitted, env, obs)
        elif name == "scenario":
            action, used_fallback = scenario_action(env, batched, planning)
            fallback += used_fallback
        else:
            action = int(choose_lookahead(env))
        assert env.mask[action]
        actions.append(str(action))
        obs, _, done, _, info = env.step(action)
        if done:
            break
    return dict(soh_loss_pp=case["state"][0] - info["soh"], delivered_wh=info["wh"], budget_wh=info["budget"],
                budget_hit=info["wh"] >= info["budget"] and not info["unsupported_stop"],
                guard_recoveries=info["guard_recoveries"], unsupported_stop=info["unsupported_stop"],
                intervals_completed=len(actions), actions="".join(actions), planning_fallbacks=fallback)


def evaluate():
    results, training_counts, context_report = [], [], []
    evaluation_rng = np.random.default_rng(SEED)
    for fold, held_out in enumerate(cells):
        study = ScheduleStudy(held_out, models[held_out], disturbances[held_out])
        train, test, excluded = all_cases(study)
        assert all(cell_ids[item["index"]] != held_out for item in train)
        assert all(cell_ids[item["index"]] == held_out for item in test)
        assert all(cell_ids[index] != held_out for item in train + test for index in item["templates"])
        context_report.append((held_out, len(train), len(test), len(excluded)))
        if not test:
            continue
        env = DischargeEnv(study, train)
        batched = BatchedStudy(study)
        check_batched_model(env, batched, train[0], study.noise[0])
        training_rng = np.random.default_rng(np.random.SeedSequence([SEED, fold, 1]))
        fitted, notes = fit_fqi(collect_batch(train, study, batched, training_rng))
        training_counts.extend((held_out, *row) for row in notes)
        planning_rng = np.random.default_rng(np.random.SeedSequence([SEED, fold, 2]))
        planning_paths = study.noise[planning_rng.integers(len(study.noise), size=PLANNING_PATHS)]
        env.reset(options={"case": test[0], "noise": study.noise[0]})
        first_plan = scenario_action(env, batched, planning_paths)
        env.noise = np.full((HORIZON, 2), 1000)
        assert first_plan == scenario_action(env, batched, planning_paths)
        for case in test:
            start = int(data.iloc[case["index"]].start_cycle)
            draws = study.noise[evaluation_rng.integers(len(study.noise), size=DISTURBANCES)]
            for draw, noise in enumerate(draws):
                for policy in POLICIES:
                    agents = ppo_actors[held_out].items() if policy == "ppo" else [(0, None)]
                    for seed, actor in agents:
                        result = episode(env, case, noise, policy, actor, fitted, batched, planning_paths)
                        results.append(dict(cell_id=held_out, start_cycle=start, draw=draw, seed=seed,
                                            policy=policy, **result))
        print(f"Completed {held_out}: {len(test)} matched starts and {sum(row[1] for row in notes):,} "
              "FQI training transitions.", flush=True)
    return (pd.DataFrame(results),
            pd.DataFrame(training_counts, columns=["cell_id", "stage", "transitions", "terminal", "action_0", "action_1", "action_2"]),
            pd.DataFrame(context_report, columns=["cell_id", "train_starts", "test_starts", "excluded_starts"]))


def verify_baselines(episodes, cases):
    keys = ["cell_id", "start_cycle", "draw", "policy", "seed"]
    reference = read_table("test_episodes", pilot, dtype={"actions": "string"})
    reference = reference.loc[reference.policy.isin(["lookahead", "ppo"])].set_index(keys, verify_integrity=True).sort_index()
    restored = episodes.loc[episodes.policy.isin(["lookahead", "ppo"])].set_index(keys, verify_integrity=True).sort_index()
    pd.testing.assert_index_equal(reference.index, restored.index)
    for name in ("soh_loss_pp", "budget_wh", "delivered_wh"):
        np.testing.assert_allclose(reference[name], restored[name], rtol=0, atol=TOLERANCE)
    for name in ("budget_hit", "unsupported_stop", "guard_recoveries", "intervals_completed", "actions"):
        np.testing.assert_array_equal(reference[name], restored[name])
    assert cases.test_starts.sum() == 133
    assert cases.test_starts.sum() + cases.excluded_starts.sum() == len(forecast_predictions)


with threadpool_limits(limits=1):
    episodes, fqi_training, cases = evaluate()
verify_baselines(episodes, cases)
print("Reproduced every original PPO and lookahead episode on the same 133 starts.")

Completed XJTU_RW_battery-1: 12 matched starts and 38,526 FQI training transitions.
Completed XJTU_RW_battery-2: 22 matched starts and 34,769 FQI training transitions.
Completed XJTU_RW_battery-3: 25 matched starts and 33,877 FQI training transitions.
Completed XJTU_RW_battery-4: 17 matched starts and 36,854 FQI training transitions.
Completed XJTU_RW_battery-5: 18 matched starts and 36,009 FQI training transitions.
Completed XJTU_RW_battery-6: 14 matched starts and 37,278 FQI training transitions.
Completed XJTU_RW_battery-7: 12 matched starts and 38,594 FQI training transitions.
Completed XJTU_RW_battery-8: 13 matched starts and 37,853 FQI training transitions.
Reproduced every original PPO and lookahead episode on the same 133 starts.


In [9]:
def paired_interval(values):
    random = np.random.default_rng(SEED)
    indexes = random.integers(len(values), size=(BOOTSTRAP_DRAWS, len(values)))
    return np.quantile(values[indexes].mean(axis=1), [0.025, 0.975])


def upper_decile(values):
    count = int(np.ceil(len(values) * TAIL_FRACTION))
    return float(np.sort(np.asarray(values))[-count:].mean())


def score_forecasts(predictions):
    error = predictions[list(FORECAST_LABELS)].sub(predictions.measured_soh, axis=0).abs()
    by_cell = error.groupby(predictions.cell_id).mean()
    previous = error.recursive
    baseline = np.array([by_cell.recursive.mean(), upper_decile(previous), previous.max()])
    rows = []
    for model, values in error.items():
        paired = (by_cell[model] - by_cell.recursive).to_numpy()
        lo, hi = paired_interval(paired)
        summary = np.array([by_cell[model].mean(), upper_decile(values), values.max()])
        worst = predictions.loc[values.idxmax()]
        rows.append(dict(model=model, batteries=len(by_cell), windows=len(values),
                         mae_pp=summary[0], tail_mae_pp=summary[1], max_error_pp=summary[2],
                         mae_difference_pp=float(paired.mean()), mae_ci95_low_pp=lo, mae_ci95_high_pp=hi,
                         worst_cell=worst.cell_id, worst_start_cycle=int(worst.start_cycle),
                         improves_all=bool((summary < baseline - TOLERANCE).all())))
    frame = pd.DataFrame(rows).set_index("model")
    qualified = frame.loc[frame.improves_all].sort_values(["mae_pp", "tail_mae_pp", "max_error_pp"], kind="stable")
    selected = qualified.index[0] if len(qualified) else "recursive"
    frame["selected"] = frame.index == selected
    return frame


def score_controllers(episodes):
    outputs = []
    aggregate = ["soh_loss_pp", "delivered_wh", "budget_hit", "unsupported_stop", "planning_fallbacks"]
    per_battery = episodes.groupby(["policy", "cell_id"])[aggregate].mean()
    per_battery["tail_loss_pp"] = episodes.groupby(["policy", "cell_id"]).soh_loss_pp.agg(upper_decile)
    policy_stats = per_battery.groupby(level="policy").mean().reindex(POLICIES)
    for name, frame in episodes.groupby("policy"):
        policy_stats.loc[name, "episodes"] = len(frame)
        policy_stats.loc[name, "worst_loss_pp"] = frame.soh_loss_pp.max()

    comparisons = [("fqi", "ppo"), ("fqi", "lookahead"), ("scenario", "lookahead")]
    for name, baseline in comparisons:
        difference = per_battery.loc[name] - per_battery.loc[baseline]
        means = difference.mean()
        loss_lo, loss_hi = paired_interval(difference.soh_loss_pp.to_numpy())
        tail_lo, tail_hi = paired_interval(difference.tail_loss_pp.to_numpy())
        energy_lo, energy_hi = paired_interval(difference.budget_hit.to_numpy())
        mean_better = means.soh_loss_pp < -TOLERANCE
        tail_better = means.tail_loss_pp < -TOLERANCE
        if name == "scenario":
            loss_pass = ((mean_better and means.tail_loss_pp <= TOLERANCE)
                         or (tail_better and means.soh_loss_pp <= TOLERANCE))
        else:
            loss_pass = mean_better
        energy_pass = means.budget_hit >= -ENERGY_RATE_TOLERANCE - TOLERANCE
        support_pass = means.unsupported_stop <= TOLERANCE
        outputs.append(dict(policy=name, baseline=baseline, mean_loss_change_pp=means.soh_loss_pp,
                            mean_ci95_low=loss_lo, mean_ci95_high=loss_hi,
                            tail_loss_change_pp=means.tail_loss_pp, tail_ci95_low=tail_lo, tail_ci95_high=tail_hi,
                            budget_hit_change=means.budget_hit, budget_ci95_low=energy_lo,
                            budget_ci95_high=energy_hi, support_stop_change=means.unsupported_stop,
                            loss_pass=bool(loss_pass), energy_pass=bool(energy_pass), support_pass=bool(support_pass),
                            qualifies=bool(loss_pass and energy_pass and support_pass)))
    decision = pd.DataFrame(outputs)
    new = decision.loc[decision.baseline.eq("lookahead") & decision.qualifies, "policy"]
    best = policy_stats.loc[new].sort_values(["soh_loss_pp", "tail_loss_pp"], kind="stable").index
    policy_stats["selected"] = policy_stats.index == (best[0] if len(best) else "lookahead")
    return policy_stats, decision


forecast_metrics = score_forecasts(forecast_predictions)
controller_metrics, controller_decisions = score_controllers(episodes)
print(forecast_metrics[["mae_pp", "tail_mae_pp", "max_error_pp", "selected"]].round(4).to_string())
print(controller_metrics[["soh_loss_pp", "tail_loss_pp", "budget_hit", "unsupported_stop", "selected"]].round(4).to_string())
print(controller_decisions[["policy", "baseline", "qualifies"]].to_string(index=False))

             mae_pp  tail_mae_pp  max_error_pp  selected
model                                                   
recursive    0.6255       2.0453        2.9478     False
direct       0.6146       1.9783        2.7614     False
history      0.5984       2.0346        2.6031      True
transformer  0.6476       2.2283        3.2937     False
residual     0.5751       2.2045        2.9900     False
           soh_loss_pp  tail_loss_pp  budget_hit  unsupported_stop  selected
policy                                                                      
ppo             1.9853        7.3645      0.9961            0.0039     False
fqi             1.8996        7.2888      0.9961            0.0039     False
lookahead       1.7547        7.1017      0.9961            0.0039      True
scenario        1.7547        7.1017      0.9961            0.0039     False
  policy  baseline  qualifies
     fqi       ppo       True
     fqi lookahead      False
scenario lookahead      False


In [10]:
def make_summary():
    fmt_pct = lambda value: f"{value * 100:+.2f} pp"
    winner = forecast_metrics.index[forecast_metrics.selected][0]
    controller = controller_metrics.index[controller_metrics.selected][0]
    qualified = controller_decisions.set_index(["policy", "baseline"]).qualifies
    fqi_verdict = "FQI qualified as an improvement over PPO." if qualified.loc["fqi", "ppo"] else (
        "FQI fell short of the comparison criteria against PPO."
    )
    lines = [
        "# XJTU SOH forecasting, scenario-based lookahead and fitted Q iteration", "",
        "## Results", "",
        f"**Selected three-test forecast: {FORECAST_LABELS[winner]}.** "
        f"**Selected controller: {POLICIES[controller]}.** {fqi_verdict}", "",
        "Forecast selection uses measured reference SOH. Controller selection uses the unchanged Notebook 20 "
        "simulator, so the two comparisons answer separate questions.", "",
        "## SOH forecasts on held-out batteries", "",
        f"All rows use the same {len(forecast_predictions)} three-test windows on {len(cells)} batteries. "
        "The same cohort was used in exploratory model screening. The one-test forecast remains the Notebook 19 model.", "",
        "| Model | Mean absolute error (pp) | Worst 10% mean (pp) | Maximum (pp) | "
        "Paired MAE difference from Notebook 19 (95% interval) |",
        "| --- | ---: | ---: | ---: | ---: |",
    ]
    for model, row in forecast_metrics.iterrows():
        lines.append(f"| {FORECAST_LABELS[model]} | {row.mae_pp:.3f} | {row.tail_mae_pp:.3f} | {row.max_error_pp:.3f} | "
                     f"{row.mae_difference_pp:+.3f} ({row.mae_ci95_low_pp:+.3f} to {row.mae_ci95_high_pp:+.3f}) |")
    row = forecast_metrics.loc[winner]
    lines.extend([
        "",
        f"The largest selected-forecast error occurred in {row.worst_cell} from recorded cycle "
        f"{int(row.worst_start_cycle)}. Forecast selection requires lower mean, tail and maximum errors. "
        "The table averages the three seeds for each Transformer forecast. The result bundle includes "
        "seed-level predictions and training-only Ridge selections.", "",
        "## Discharge controller comparison", "",
        f"The evaluation reproduced every original PPO and lookahead trajectory at {int(cases.test_starts.sum())} "
        "starts. It used the same eight evaluation disturbances per start, 90% energy budget and action guard. "
        "Mean and tail controller metrics give each battery equal weight. The tail is the mean loss among "
        "the worst 10% of episodes for each battery. Modelled losses are reference-test SOH percentage points.", "",
        "| Policy | Mean loss (pp) | Worst 10% mean loss (pp) | Maximum loss (pp) | Energy budget hit | Support stops |",
        "| --- | ---: | ---: | ---: | ---: | ---: |",
    ])
    for name, row in controller_metrics.iterrows():
        lines.append(f"| {POLICIES[name]} | {row.soh_loss_pp:.3f} | {row.tail_loss_pp:.3f} | {row.worst_loss_pp:.3f} | "
                     f"{row.budget_hit:.2%} | {row.unsupported_stop:.2%} |")
    scenario = episodes.loc[episodes.policy.eq("scenario")]
    keys = ["cell_id", "start_cycle", "draw"]
    original = episodes.loc[episodes.policy.eq("lookahead")].set_index(keys).actions
    compared = scenario.set_index(keys).actions
    changed = int(compared.ne(original.reindex(compared.index)).sum())
    lines.extend([
        "",
        f"The scenario planner fell back to the original lookahead for {int(scenario.planning_fallbacks.sum())} of "
        f"{int(scenario.intervals_completed.sum())} decisions. It changed the executed schedule on "
        f"{changed} of {len(scenario)} matched evaluation episodes. Both learned policies face the same action mask. "
        "FQI uses simulated transition batches from the training batteries. The planning and evaluation draw "
        "streams have separate seeds.", "",
        "## Paired controller decisions", "",
        "| Comparison | Mean loss change (95% interval) | Tail loss change (pp) | "
        "Budget hit change | Support stop change | Meets comparison criteria |",
        "| --- | ---: | ---: | ---: | ---: | --- |",
    ])
    for test in controller_decisions.itertuples(index=False):
        passed = "Yes" if test.qualifies else "No"
        lines.append(f"| {POLICIES[test.policy]} vs {POLICIES[test.baseline]} | "
                     f"{test.mean_loss_change_pp:+.3f} ({test.mean_ci95_low:+.3f} to {test.mean_ci95_high:+.3f}) | "
                     f"{test.tail_loss_change_pp:+.3f} | {fmt_pct(test.budget_hit_change)} | "
                     f"{fmt_pct(test.support_stop_change)} | {passed} |")
    lines.extend([
        "",
        "Paired intervals resample held-out batteries. The control comparison allows up to a one percentage-point "
        "reduction in the energy budget hit rate and requires no increase in support stops. The scenario "
        "controller must improve average or tail SOH loss while the other stays at least as good. FQI's "
        "comparison uses lower average loss. The benchmark covers 30 operating cycles of modelled discharge "
        "and does not estimate real-world lifetime gains.", "",
        "[Fitted Q Iteration research](https://www.jmlr.org/papers/v6/ernst05a.html) · "
        "[Scenario-based MPC research](https://arxiv.org/abs/1307.5640)", "",
    ])
    return "\n".join(lines)


def draw_score_chart(path):
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(10.6, 4), layout="constrained")
    left, right = axes
    left_labels = ["Recursive Ridge", "Direct Ridge", "Direct Ridge + history", "Compact Transformer",
                   "Residual Transformer"]
    left.barh(left_labels, forecast_metrics.mae_pp, color="#336f9c")
    left.set(xlabel="SOH mean absolute error (pp)", title="Three-test forecast")
    right.barh(list(POLICIES.values()), controller_metrics.soh_loss_pp, color="#288879")
    right.set(xlabel="Modelled mean SOH loss (pp)", title="Held-out controller simulation")
    for ax in axes:
        ax.invert_yaxis()
        ax.grid(axis="x", alpha=.18)
    fig.savefig(path, dpi=145)
    plt.close(fig)


output.mkdir(parents=True, exist_ok=True)
tables = {
    "forecast_predictions": forecast_predictions, "forecast_metrics": forecast_metrics.reset_index(),
    "forecast_seeds": forecast_seed_results, "ridge_choices": ridge_selection,
    "controller_metrics": controller_metrics.reset_index(), "controller_comparisons": controller_decisions,
    "controller_episodes": episodes, "fqi_training": fqi_training, "controller_cases": cases,
}
for name, table in tables.items():
    table.to_csv(output / f"{name}.csv", index=False)
metadata = {
    "forecast_horizon_tests": HORIZON, "forecast_seeds": FORECAST_SEEDS, "ridge_alphas": RIDGE_ALPHAS,
    "transformer_epochs": TRANSFORMER_EPOCHS, "fqi_training_paths_per_start": TRAIN_BRANCHES, "fqi_trees": 64,
    "fqi_max_depth": 6, "fqi_min_samples_leaf": 5, "scenario_planning_paths": PLANNING_PATHS,
    "scenario_required_energy_probability": PLANNING_ENERGY_PROBABILITY, "scenario_tail_fraction": TAIL_FRACTION,
    "test_seed": SEED, "test_disturbances_per_start": DISTURBANCES, "bootstrap_draws": BOOTSTRAP_DRAWS,
    "energy_budget_fraction": ENERGY_FRACTION, "energy_hit_rate_tolerance": ENERGY_RATE_TOLERANCE,
}
(output / "run_config.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
(output / "summary.md").write_text(make_summary(), encoding="utf-8")
draw_score_chart(output / "model_comparison.png")
files = [*(f"{name}.csv" for name in tables), "run_config.json", "summary.md", "model_comparison.png"]
archive_path = output.with_suffix(".zip")
with ZipFile(archive_path, "w", ZIP_DEFLATED) as bundle:
    for name in files:
        bundle.write(output / name, arcname=f"{output.name}/{name}")
print(f"Report: {output / 'summary.md'}")
print(f"Download: {archive_path}")

Report: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_xjtu_forecast_fqi_results/summary.md
Download: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_xjtu_forecast_fqi_results.zip
